# Nonlinear 3–3–3 parameter sweep

Train the network for every combination of `random_state_M`, `random_state`, and `alpha_scale_nonlin`. Apart from those sweep axes, the fixed 3–3–3 structure, and the nonlinear resistance update rule, simulation parameters come from `config.py`.

In [ ]:
from __future__ import annotations

import importlib
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray
from typing import Any, Optional, cast
from matplotlib.colors import LogNorm

import config
import file_funcs
import functions
import plot_funcs
import statistical_analysis as statistics
from Big_Class import Big_Class
from Network_State import Network_State
from Network_Structure import Network_Structure
from Supervisor_Class import Supervisor
from User_Variables import User_Variables

importlib.reload(config)
CFG = config.CFG

import colors

colors_lst, red, cmap = colors.color_scheme()

## Define the sweep

Edit only these three arrays to choose the runs. `alpha_scale_nonlin_values` controls the nonlinear multiplier; the base `alpha` still comes from `config.py`.

In [ ]:
random_state_M_values = np.array([36, 37, 38, 39, 40], dtype=int)
random_state_values = np.array([52, 53, 54, 55, 56], dtype=int)
alpha_scale_nonlin_values = np.array([45, 47, 49, 50, 51, 52, 53, 55, 57, 59, 61], dtype=float)

output_dir = Path("multiple_Nin_Nout_outputs_July21")
output_dir.mkdir(parents=True, exist_ok=True)

average_final_loss = np.full(
    (len(random_state_M_values), len(random_state_values), len(alpha_scale_nonlin_values)),
    np.nan,
    dtype=float,
)
minimum_moving_mean_loss = np.full_like(average_final_loss, np.nan)
average_final_loss.shape, minimum_moving_mean_loss.shape

In [ ]:
def build_run(
    random_state_M: int, random_state: int, alpha_scale_nonlin: float
) -> tuple[config.ExperimentConfig, Big_Class, Network_State]:
    """Build one nonlinear 3–3–3 run, inheriting all other values from CFG."""
    n_inputs = n_outputs = n_intermediate = 3
    m_values = functions.random_gen_M(random_state_M, n_inputs * n_outputs)
    run_cfg = replace(
        CFG,
        Strctr=replace(
            CFG.Strctr, Nin=n_inputs, Nout=n_outputs, Ninter=n_intermediate
        ),
        Variabs=replace(CFG.Variabs, R_update="deltaR_propto_dp_nonlin"),
        Sprvsr=replace(
            CFG.Sprvsr,
            M_values=m_values,
            random_state_M=int(random_state_M),
            random_state=int(random_state),
            alpha_scale_nonlin=float(alpha_scale_nonlin),
        ),
    )

    structure = Network_Structure(run_cfg)
    structure.build_incidence(structure.net_type)
    big_class = Big_Class(structure)
    variables = User_Variables(run_cfg)
    supervisor = Supervisor(run_cfg, structure, variables)
    big_class.add_Variabs(variables)
    big_class.add_Sprvsr(supervisor)

    if supervisor.training_scheme in {"GD_like", "Adaline"}:
        structure.build_inverse_incidence()
        if supervisor.training_scheme == "GD_like":
            structure.build_RM()
    if supervisor.training_scheme == "Adaline" and structure.Ninter > 0:
        fict_cfg = replace(run_cfg, Strctr=replace(run_cfg.Strctr, Ninter=0, net_type="FC"))
        fict_structure = Network_Structure(fict_cfg)
        fict_structure.build_incidence(fict_structure.net_type)
        fict_structure.build_inverse_incidence()
        big_class.add_Strctr_fict(fict_structure)

    state = Network_State(run_cfg, big_class)
    state.initiate_resistances(big_class)
    if supervisor.task_type == "Iris_classification":
        state.initiate_accuracy_vec(big_class)
    big_class.add_State(state)
    return run_cfg, big_class, state


def train_loop(state: Network_State, big_class: Big_Class) -> Network_State:
    """Train one run using the current notebook/Main training sequence."""
    supervisor = big_class.Sprvsr
    variables = big_class.Variabs
    structure = big_class.Strctr

    for i in range(supervisor.iterations):
        if supervisor.use_p_tag and supervisor.noise_to_extra:
            k = 2 * (i // supervisor.stay_sample) + 1
            if not i % 4:
                k -= 1
        elif supervisor.use_p_tag:
            k = (i // supervisor.stay_sample) * 2 + i % 2
        else:
            k = i // supervisor.stay_sample

        add_extra_noise = not (i + 1) % 4 and supervisor.noise_to_extra
        state.draw_p_in_and_desired(supervisor, k, noise_to_extra=add_extra_noise)
        state.solve_flow_given_modality(
            big_class, "measure", noise_to_extra=add_extra_noise
        )
        if supervisor.include_Power:
            state.calc_Power_norm(big_class)

        if not i % 2 and supervisor.use_p_tag:
            continue

        state.t += 1
        state.calc_loss(big_class)
        if not (i + 1) % 4 and supervisor.access_interNodes:
            state.update_inter(big_class)
        else:
            if supervisor.training_scheme in {"GD_like", "Adaline"}:
                state.calc_update_vals_vec(big_class)
            state.update_input(big_class)
            state.update_output(big_class)

        update_iterations = 6 if structure.net_type == "beads" else 1
        for _ in range(update_iterations):
            state.solve_flow_given_modality(
                big_class, "update", access_inters=supervisor.access_interNodes
            )
            state.update_Rs(big_class)

        if supervisor.anneal and state.t < supervisor.iterations:
            supervisor.anneal_alpha(state)

    denominator = (
        supervisor.y_train
        if supervisor.task_type == "Regression"
        else np.array([[np.mean(state.targets_mat)]])
    )
    losses = np.asarray(state.loss_in_t)
    if supervisor.loss_type == "MAE":
        state.loss_scalar_in_t = np.mean(np.mean(np.abs(losses), axis=1), axis=1)
        state.loss_scalar_in_t /= np.mean(np.abs(denominator), axis=1)
    elif supervisor.loss_type == "MSE":
        state.loss_scalar_in_t = np.mean(np.mean(np.square(losses), axis=1), axis=1)
        state.loss_scalar_in_t /= np.mean(np.square(denominator), axis=1)
    else:
        raise ValueError(f"Unknown loss type: {supervisor.loss_type}")
    return state


def value_for_filename(value: float) -> str:
    """Return a compact filename-safe representation of a numeric value."""
    return f"{value:.12g}".replace("-", "m").replace(".", "p")

## Run and save

Each run writes one training CSV and one `plot_importants` PNG. Both 3D loss arrays use axis order `(random_state_M, random_state, alpha_scale_nonlin)`. `average_final_loss` contains the mean of the final 40 scalar-loss samples, via `statistics.final_err`; `minimum_moving_mean_loss` contains the minimum value of the 400-step moving-mean loss. The compressed NPZ is refreshed after every run so partial sweeps remain usable.

In [ ]:
total_runs = average_final_loss.size
run_number = 0

for m_idx, random_state_M in enumerate(random_state_M_values):
    for data_idx, random_state in enumerate(random_state_values):
        for alpha_idx, alpha_scale_nonlin in enumerate(alpha_scale_nonlin_values):
            run_number += 1
            run_label = (
                f"random_state_M-{int(random_state_M)}"
                f"_random_state-{int(random_state)}"
                f"_alpha-{value_for_filename(float(alpha_scale_nonlin))}"
            )
            print(f"[{run_number}/{total_runs}] {run_label}")

            run_cfg, BigClass, State = build_run(
                int(random_state_M), int(random_state), float(alpha_scale_nonlin)
            )
            State = train_loop(State, BigClass)

            average_final_loss[m_idx, data_idx, alpha_idx] = statistics.final_err(
                State.loss_scalar_in_t
            )
            loss_movmean = statistics.mov_ave(State.loss_scalar_in_t, 250)
            min_mean_loss_ij = loss_movmean[loss_movmean == np.min(loss_movmean)]
            minimum_moving_mean_loss[m_idx, data_idx, alpha_idx] = float(
                min_mean_loss_ij[0]
            )
            csv_path = output_dir / f"training_{run_label}.csv"
            plot_stem = output_dir / f"plot_importants_{run_label}"
            file_funcs.export_importants_csv(State, csv_path)
            plot_funcs.plot_importants(
                BigClass,
                movmean_loss=True,
                include_network=False,
                node_labels=False,
                save=True,
                name=str(plot_stem),
            )
            plt.close("all")

            np.savez_compressed(
                output_dir / "average_final_loss_3d.npz",
                average_final_loss=average_final_loss,
                minimum_moving_mean_loss=minimum_moving_mean_loss,
                random_state_M_values=random_state_M_values,
                random_state_values=random_state_values,
                alpha_scale_nonlin_values=alpha_scale_nonlin_values,
            )

average_final_loss, minimum_moving_mean_loss

In [ ]:
log_scale = True
# log_scale = False
vmin: Optional[float] = None
vmax: Optional[float] = None
if log_scale:
    loss_max = 1
    loss_min = 1e-3
    # Set color normalization using logarithmic scale
    log_norm_factory: Any = LogNorm
    norm = log_norm_factory(vmin=loss_min, vmax=loss_max)
else:
    vmin = 0
    vmax = 0.1
    norm = None


for i in range(len(random_state_M_values)):
    fig, raw_ax = plt.subplots()
    ax = cast(plt.Axes, raw_ax)
    ax.imshow(average_final_loss[i,:,:], cmap=cmap, norm=norm,
              vmin=(vmin if not log_scale else None), vmax=(vmax if not log_scale else None), origin='lower',)
    plt.show()

for i in range(len(random_state_M_values)):
    fig, raw_ax = plt.subplots()
    ax = cast(plt.Axes, raw_ax)
    ax.imshow(minimum_moving_mean_loss[i,:,:], cmap=cmap, norm=norm,
              vmin=(vmin if not log_scale else None), vmax=(vmax if not log_scale else None), origin='lower',)
    plt.show()


In [ ]:
ave_afo_alpha = np.mean(np.mean(average_final_loss, axis=0), axis=0)
err_ave = np.std(np.std(average_final_loss, axis=0), axis=0)

fig, raw_ax = plt.subplots()
ax = cast(plt.Axes, raw_ax)
ax.errorbar(
    alpha_scale_nonlin_values,
    ave_afo_alpha,
    yerr=err_ave,
    marker="o",
    capsize=4,
)

ax.set_xlabel(r"$\alpha_{\mathrm{scale,nonlin}}$")
ax.set_ylabel("average final loss")
ax.set_yscale("log")
plt.show()

min_afo_alpha = np.mean(np.mean(minimum_moving_mean_loss, axis=0), axis=0)
err_min = np.std(np.std(minimum_moving_mean_loss, axis=0), axis=0)

fig, raw_ax = plt.subplots()
ax = cast(plt.Axes, raw_ax)
ax.errorbar(
    alpha_scale_nonlin_values,
    min_afo_alpha,
    yerr=err,
    marker="o",
    capsize=4,
)

ax.set_xlabel(r"$\alpha_{\mathrm{scale,nonlin}}$")
ax.set_ylabel("Minimum moving-mean loss")
ax.set_yscale("log")
plt.show()